In [253]:
import numba

In [254]:
import math
import random
import numpy as np

def randn(*n):
    return np.random.randn(*n) * 2 - 1
    # return [random.random() * 2 - 1 for _ in range(n)]


In [255]:
w, h = 2, 2
s = 3
k = 2

# w, h = 28, 28
# s = 300
# k = 10

In [256]:
wh = w * h

# Forward

In [257]:
X = randn(wh)
X

array([-1.66006966,  0.54093981, -6.31395412,  0.862226  ])

In [258]:
# layer_hidden_weights = [randn(wh) for _ in range(s)]
layer_hidden_weights = randn(s, wh)
layer_hidden_bias = randn(s)


# @numba.jit()
def relu_forward(x: float):
    return max(0.0, x)

# @numba.jit()
def layer_hidden_forward(x):
    y = [0.0] * s
    for i in range(s):
        u = 0.0

        for j in range(wh):
            u += x[j] * layer_hidden_weights[i][j]
        u += layer_hidden_bias[i]

        # NOTE: 
        # we may separate passes and run relu on whole `y` vector. 
        #   like `y = relu_forward(y)`
        # but it looks like overcomplication for now
        y[i] = relu_forward(u)

    return y

In [259]:
H = layer_hidden_forward(X)
H

[0.0, np.float64(6.140180141688851), 0.0]

In [260]:

# layer_output_weights = [randn(s) for _ in range(k)]
layer_output_weights = randn(k, s)
layer_output_bias = randn(k)

# @numba.jit()
def softmax_forward(x):
    y = [0.0] * k

    e_max = max(x)
    e_sum = 0
    for i in range(k):
        e_sum += math.exp(x[i] - e_max)
    
    for i in range(k):
        e = math.exp(x[i] - e_max)
        y[i] = e / e_sum

    return y

# @numba.jit()
def layer_output_forward(x):
    y = [0.0] * k
    for i in range(k):
        u = 0

        for j in range(s):
            u += x[j] * layer_output_weights[i][j]
        u += layer_output_bias[i]

        y[i] = u

    return y

In [261]:
O = layer_output_forward(H)
O

[np.float64(-4.752403860368103), np.float64(4.7798579369719265)]

In [262]:
softmax_forward(O)

[7.247026122697278e-05, 0.999927529738773]

In [263]:
# Y = [0.0] * k
Y = np.zeros(k)
Y[random.randint(0, k-1)] = 1.0

# @numba.jit()
def layer_loss_forward(x, y):
    x = softmax_forward(x)

    err_sum = 0.0
    for i in range(k):
        # math.log(x[i]) `ValueError: math domain error` when `x[i] == 0.0`
        if x[i] == 0.0:
            if y[i] == 0.0:
                v = 0.0
            else:
                v = -np.inf
        else:
            v = y[i] * math.log(x[i])

        err_sum += v

    return -err_sum

In [264]:
E = layer_loss_forward(
    O, Y
)
E, Y

(np.float64(7.2472887323289e-05), array([0., 1.]))

In [265]:
layer_loss_forward(
    layer_output_forward(
        layer_hidden_forward(X)
    ),
    Y
)


np.float64(7.2472887323289e-05)

# Backward

In [266]:
import torch


In [267]:
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(2.0, requires_grad=True)

f = a ** 2 * b
f.backward()

a.grad == (2 * 3 * 2), b.grad == (3 ** 2)

(tensor(True), tensor(True))

In [268]:
layer_hidden_weights

array([[ 1.92655161, -2.25724197, -0.67638382,  1.38000587],
       [-1.84554663,  0.45120404, -1.19722764, -1.0321372 ],
       [-0.67888574,  2.05757136,  0.35024196,  0.8326526 ]])

In [269]:
layer_output_weights_t = torch.tensor(layer_output_weights, requires_grad=True)
layer_output_bias_t = torch.tensor(layer_output_bias, requires_grad=True)

x = torch.ones(s) * 2 

u = 0.0

for j in range(s):
    u += x[j] * layer_output_weights_t[0][j]
u += layer_output_bias_t[0]

u.backward()


In [270]:
layer_output_bias_t.grad

tensor([1., 0.], dtype=torch.float64)

In [271]:
layer_output_weights_t.grad

tensor([[2., 2., 2.],
        [0., 0., 0.]], dtype=torch.float64)

In [272]:
torch.manual_seed(0)
v = torch.ones(s, requires_grad=True)
v = torch.rand(s, requires_grad=True)

y = torch.zeros(s)
y[0] = 1.0
y = torch.rand(s)

e = v.exp()
sm = e.sum()
u = e / sm

m = y * torch.log(u)
L = - (m.sum())

L.backward()

# u, m, y, L

In [273]:
v.grad

tensor([ 0.2285,  0.1658, -0.3943])

In [274]:
u * y.sum() - y # dL/dv

tensor([ 0.2285,  0.1658, -0.3943], grad_fn=<SubBackward0>)

In [275]:
torch.manual_seed(0)
layer_output_weights_t = torch.rand((k, s), requires_grad=True)
layer_output_bias_t = torch.rand((k), requires_grad=True)

x = torch.ones(s)
x = torch.rand(s)


v_elements = []

for i in range(k):
    v_i = 0 
    for j in range(s):
        v_i += x[j] * layer_output_weights_t[i][j]
    v_i += layer_output_bias_t[i]

    v_elements.append(v_i)

v = torch.stack(v_elements)

# v = torch.mv(layer_output_weights_t, x) + layer_output_bias_t
v.retain_grad()


e = v.exp()
sm = e.sum()
u = e / sm

y = torch.zeros(k)
y[0] = 1.0
y = torch.rand(k)

m = y * torch.log(u)
L = - (m.sum())

L.backward()

u, m, y, L

(tensor([0.4652, 0.5348], grad_fn=<DivBackward0>),
 tensor([-0.3074, -0.0140], grad_fn=<MulBackward0>),
 tensor([0.4017, 0.0223]),
 tensor(0.3214, grad_fn=<NegBackward0>))

In [276]:
layer_output_weights_t.grad, layer_output_bias_t.grad

(tensor([[-0.0932, -0.1293, -0.0713],
         [ 0.0932,  0.1293,  0.0713]]),
 tensor([-0.2044,  0.2044]))

In [277]:
torch.outer(
    u * y.sum() - y, x
) # dL/dw_o


tensor([[-0.0932, -0.1293, -0.0713],
        [ 0.0932,  0.1293,  0.0713]], grad_fn=<MulBackward0>)

In [278]:
u * y.sum() - y # dL/db_o

tensor([-0.2044,  0.2044], grad_fn=<SubBackward0>)

In [279]:
v.grad

tensor([-0.2044,  0.2044])

In [280]:
torch.manual_seed(4)

layer_hidden_weights_t = torch.normal(0, 0.35, (s, wh), requires_grad=True)
layer_hidden_bias_t = torch.normal(0.01, 0.05, (s, ), requires_grad=True)


x = torch.rand(wh)

p_elements = []
for i in range(s):
    u = 0.0

    for j in range(wh):
        u += x[j] * layer_hidden_weights_t[i][j]
    u += layer_hidden_bias_t[i]

    p_elements.append(u)

p = torch.stack(p_elements)
p.retain_grad()


layer_output_weights_t = torch.rand((k, s), requires_grad=True)
layer_output_bias_t = torch.rand((k), requires_grad=True)

# h = torch.ones(s)
# h = torch.rand(s)

h_elements = []
for i in range(s):
    p_ = max(torch.tensor(0.0, requires_grad=True), p[i])
    h_elements.append(p_)
h = torch.stack(h_elements)
h.retain_grad()


v_elements = []
for i in range(k):
    v_i = 0 
    for j in range(s):
        v_i += h[j] * layer_output_weights_t[i][j]
    v_i += layer_output_bias_t[i]

    v_elements.append(v_i)

v = torch.stack(v_elements)

# v = torch.mv(layer_output_weights_t, x) + layer_output_bias_t
v.retain_grad()


e = v.exp()
sm = e.sum()
u = e / sm

y = torch.zeros(k)
y[0] = 1.0
y = torch.rand(k)

m = y * torch.log(u)
L = - (m.sum())

L.backward()

u, m, y, L

(tensor([0.6320, 0.3680], grad_fn=<DivBackward0>),
 tensor([-0.1350, -0.9208], grad_fn=<MulBackward0>),
 tensor([0.2943, 0.9210]),
 tensor(1.0558, grad_fn=<NegBackward0>))

In [281]:
layer_output_weights_t.grad, layer_output_bias_t.grad

(tensor([[ 0.3419,  0.0000,  0.3239],
         [-0.3419,  0.0000, -0.3239]]),
 tensor([ 0.4738, -0.4738]))

In [282]:
# dL/dw_o
torch.outer(
    u * y.sum() - y, h
), u * y.sum() - y # dL/db_o

(tensor([[ 0.3419,  0.0000,  0.3239],
         [-0.3419, -0.0000, -0.3239]], grad_fn=<MulBackward0>),
 tensor([ 0.4738, -0.4738], grad_fn=<SubBackward0>))

In [283]:
h.grad

tensor([-0.1549,  0.0893,  0.0156])

In [284]:
# (u * y.sum() - y) @ layer_output_weights_t # dL/dh
v.grad @ layer_output_weights_t # dL/dh

tensor([-0.1549,  0.0893,  0.0156], grad_fn=<SqueezeBackward4>)

In [285]:
p

tensor([ 0.7216, -0.3860,  0.6837], grad_fn=<StackBackward0>)

In [286]:
p.grad 

tensor([-0.1549,  0.0000,  0.0156])

In [287]:
(v.grad @ layer_output_weights_t) * (p > 0) # dL/dp

tensor([-0.1549,  0.0000,  0.0156], grad_fn=<MulBackward0>)

In [288]:
layer_hidden_bias_t.grad

tensor([-0.1549,  0.0000,  0.0156])

In [289]:
p.grad # dL/db_h

tensor([-0.1549,  0.0000,  0.0156])

In [290]:
layer_hidden_weights_t.grad

tensor([[-0.0465, -0.1249, -0.1490, -0.0337],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0047,  0.0125,  0.0150,  0.0034]])

In [291]:
torch.outer(
    p.grad, x
) # dL/dw_h

tensor([[-0.0465, -0.1249, -0.1490, -0.0337],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0047,  0.0125,  0.0150,  0.0034]])

In [ ]:
p.grad

In [319]:
torch.manual_seed(4)

layer_hidden_weights_t = torch.normal(0, 0.35, (s, wh), requires_grad=True)
layer_hidden_bias_t = torch.normal(0.01, 0.05, (s, ), requires_grad=True)
layer_output_weights_t = torch.rand((k, s), requires_grad=True)
layer_output_bias_t = torch.rand((k), requires_grad=True)


memo = {}

L_ = 2
L = torch.tensor(0.0, requires_grad=True)
for img in range(L_):
    x = torch.rand(wh)
    # y = torch.zeros(k)
    # y[0] = 1.0
    y = torch.rand(k)


    p_elements = []
    for i in range(s):
        u_i = 0.0

        for j in range(wh):
            u_i += x[j] * layer_hidden_weights_t[i][j]
        u_i += layer_hidden_bias_t[i]

        p_elements.append(u_i)
    p = torch.stack(p_elements)
    p.retain_grad()


    h_elements = []
    for i in range(s):
        p_ = max(torch.tensor(0.0, requires_grad=True), p[i])

        h_elements.append(p_)
    h = torch.stack(h_elements)
    h.retain_grad()


    v_elements = []
    for i in range(k):
        v_i = 0

        for j in range(s):
            v_i += h[j] * layer_output_weights_t[i][j]
        v_i += layer_output_bias_t[i]

        v_elements.append(v_i)
    v = torch.stack(v_elements)
    v.retain_grad()


    e = v.exp()
    sm = e.sum()
    u = e / sm


    m = y * torch.log(u)
    L = L - (m.sum())

    # memo
    memo[img] = x, p, h, v, u, y


L = L / L_

L.backward()

In [320]:
layer_hidden_weights_t.grad, layer_hidden_bias_t.grad, layer_output_weights_t.grad, layer_output_bias_t.grad

(tensor([[0.0269, 0.0209, 0.0234, 0.0226],
         [0.0907, 0.0313, 0.0389, 0.1161],
         [0.0388, 0.0302, 0.0339, 0.0326]]),
 tensor([0.0376, 0.1178, 0.0544]),
 tensor([[ 0.1430,  0.1047,  0.3978],
         [-0.1430, -0.1047, -0.3978]]),
 tensor([ 0.4524, -0.4524]))

In [ ]:
dLdb_h = 0
dLdw_h = 0
dLdb_o = 0
dLdw_o = 0

for img in range(L_):
    x, p, h, v, u, y = memo[img]

    dLdv = u * y.sum() - y
    dLdp = (dLdv @ layer_output_weights_t) * (p > 0)

    dLdb_h += dLdp
    dLdw_h += torch.outer(dLdp, x)
    dLdb_o += dLdv
    dLdw_o += torch.outer(dLdv, h)


dLdw_h / L_, dLdb_h / L_, dLdw_o / L_, dLdb_o / L_

(tensor([[0.0269, 0.0209, 0.0234, 0.0226],
         [0.0907, 0.0313, 0.0389, 0.1161],
         [0.0388, 0.0302, 0.0339, 0.0326]], grad_fn=<DivBackward0>),
 tensor([0.0376, 0.1178, 0.0544], grad_fn=<DivBackward0>),
 tensor([[ 0.1430,  0.1047,  0.3978],
         [-0.1430, -0.1047, -0.3978]], grad_fn=<DivBackward0>),
 tensor([ 0.4524, -0.4524], grad_fn=<DivBackward0>))